# 实现线性回归的简单版本
调用pytorch的api实现线性回归

## 生成数据集

In [ ]:
import torch
from torch.utils import data
import numpy as np
from d2l import torch as d2l

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)  # features对应特征X，labels对应生成的标签


c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 读取数据集

构造一个pytorch数据迭代器

In [ ]:
def load_array(data_arrays, batch_size, is_train=True): # is_train=True表示希望数据迭代器对象在每个迭代周期内打乱数据（训练阶段）
    """将数据集转换为pytorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)  # 解包，Dataset会返回一个迭代器对象
# 解包操作会将列表解包为两个独立的参数，传入TensorDataset的多个独立张量会把他们按照索引对齐（要求张量的第一维长度一致）
    return data.DataLoader(dataset, batch_size, shuffle=is_train) # 表示训练阶段，需要打乱数据的顺序
# 这个函数最后返回的是一个可迭代对象，支持使用for循环遍历数据（每次返回一个批次），但是不支持直接调用next方法
batch_size = 10
data_iter = load_array((features, labels), batch_size)
# 测试，使用iter构造python迭代器，使用next方法从迭代器中获取第一项
next(iter(data_iter))

[tensor([[ 0.7435,  0.2644],
         [ 1.9074,  0.5100],
         [-1.7769, -0.4066],
         [ 0.9384,  2.0032],
         [-0.4744,  0.8838],
         [ 0.2896, -0.8770],
         [ 0.4385, -0.3768],
         [ 1.5838, -0.6703],
         [-0.9057, -0.0650],
         [ 0.1908,  1.1704]]),
 tensor([[ 4.7928],
         [ 6.2812],
         [ 2.0139],
         [-0.7284],
         [ 0.2305],
         [ 7.7591],
         [ 6.3555],
         [ 9.6369],
         [ 2.6185],
         [ 0.5931]])]

## 定义模型

在pytorch中，全连接层在Linear类中定义，将两个参数传入到nn.Linear类中，第一个指定输入特征形状，第二个参数指定输出特征形状  
nn.sequential类是一个容器，用于按顺序堆叠神经网络，将传入的层按照顺序连接起来形成一个完整的神经网络
nn.Linear类是一个全连接层，实现了线性回归的数学公式：
它实现了线性回归的数学公式：$ \hat{y} = Xw + b $  
传入（2， 1）内部会自动初始化权重$w$为（2， 1），偏置项$b$为（1，）

In [4]:
from torch import nn
net = nn.Sequential(nn.Linear(2, 1)) # 创建一个简单的线性回归模型，输入层有2个特征，输出层有1个特征

## 初始化模型参数

在这里指定每个权重参数从正态分布$N(0, 0.01)$中随机采样，偏置参数项为0

In [5]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

除了上面那样写之外，还可以使用nn.init类中的方法来初始化模型参数
```import torch.nn.init as init
init.normal_(net[0].weight, mean=0, std=0.01)
init.constant_(net[0].bias, val=0)
```

## 定义损失函数
均方误差对应的函数是MSELoss类

In [6]:
loss = nn.MSELoss()

## 定义优化算法

In [8]:
trainer = torch.optim.SGD(net.parameters(), lr = 0.03)

net.parameters() 是 PyTorch 模型的一个方法，用于获取模型中所有可训练的参数，返回的是一个迭代器

## 训练

In [9]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)  # net = nn.Sequential(nn.Linear(2, 1))，net(X)等价于调用net.forward(X),将X传入第一层
        trainer.zero_grad()  # 在方向传播算梯度之前需要先将梯度清零，否则梯度会累加
        l.backward()  # 计算梯度
        trainer.step()  # 跟新梯度

    l = loss(net(features), labels)
    print(f'epoch{epoch+1}, loss{l:f}')


epoch1, loss0.000283
epoch2, loss0.000099
epoch3, loss0.000099


## Practice

* 取小批量损失的平均值替换小批量的总损失，怎么更改学习率？  
MSE损失函数：$loss = \frac{1}{n} \sum_{i=1}^{n} \frac{1}{2} (\hat{y}^{(i)} - y^{(i)})^2$
梯度更新规则：  
计算梯度$ g \leftarrow \partial_{(w,b)} \frac{1}{|\beta|} \sum_{i \in \beta } l(X^{(i)}, y^{(i)},w,b)$  
更新参数$(w, b) \leftarrow (w, b) - \eta g$

*为了配平系数，学习率 $\times$ batchsize*

* 用huber损失函数替换MSE损失函数，怎么做？

In [3]:
import torch
from torch.utils import data
import numpy as np
from d2l import torch as d2l
from torch import nn

true_w = torch.tensor([2, -3.4])
true_b = 4.2
current_lr = 0.03
batch_size = 10
w_grad = [] # 存放w的梯度
b_grad = [] # 存放b的梯度

features, labels = d2l.synthetic_data(true_w, true_b, 1000) # 生成数据集，其中features是特征矩阵X，labels是目标变量y

# 读取数据
def load_array(data_arrays, batch_size, is_train=True):
    dataset = data.TensorDataset(*data_arrays)  # 使用data方法，解包，传入data_arrays解包之后得到的多个独立张量并把他们按照索引对齐,得到的是一个TensorDataset对象
# dataset[i] = (feature[i], label[i]) ,可以这样通过索引访问
    return data.DataLoader(dataset, batch_size, shuffle=is_train)   # 生成一个训练的数据集，batch_size表示每个批次的样本数，shuffle表示是否打乱数据，返回的是一个可迭代对象

# 定义模型
net = nn.Sequential(nn.Linear(2, 1))
# 定义损失函数
Loss = nn.HuberLoss(delta=1.0) # delta是huber损失函数的参数，用于控制损失函数的平滑度，默认值为1.0
# 定义优化算法
trainer = torch.optim.SGD(net.parameters(), lr=current_lr)

# 开始训练
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in load_array((features, labels), batch_size):
        l = Loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        w_grad.append(net[0].weight.grad.clone().detach().numpy())
        b_grad.append(net[0].bias.grad.clone().detach().numpy())
        trainer.step() # 更新参数w和b,梯度下降的具体过程

    print(f'epoch {epoch+1}, loss: {l:f}')
print(w_grad)
print(b_grad)




epoch 1, loss: 2.948833
epoch 2, loss: 0.617123
epoch 3, loss: 0.005681
[array([[-0.33629507,  0.40272292]], dtype=float32), array([[-0.08238262,  0.15730032]], dtype=float32), array([[-0.1347777 , -0.10417458]], dtype=float32), array([[-0.36117023,  0.23354813]], dtype=float32), array([[-0.34529546,  0.31070825]], dtype=float32), array([[-0.05912677,  0.71267956]], dtype=float32), array([[-0.43752912,  0.40458727]], dtype=float32), array([[0.15007345, 0.56334007]], dtype=float32), array([[-0.00694396,  0.3970249 ]], dtype=float32), array([[-0.38625455,  0.8764596 ]], dtype=float32), array([[-0.60016096,  0.6238105 ]], dtype=float32), array([[0.00628097, 0.378014  ]], dtype=float32), array([[-0.20423293,  0.14777032]], dtype=float32), array([[0.08577198, 0.1117435 ]], dtype=float32), array([[-0.8452   ,  0.7577022]], dtype=float32), array([[0.15216757, 0.15226592]], dtype=float32), array([[-0.40519693,  0.5628039 ]], dtype=float32), array([[0.37868267, 0.68837327]], dtype=float32), arr

* 如何访问线性回归的梯度呢？
这里有两个注意点：  
1. append方法是存的是指针，而不是值。在 PyTorch 中，net[0].weight.grad 是一个常驻内存的 Tensor 对象。
如果你直接写 w_grads.append(net[0].weight.grad)，你其实只是把这个 Tensor 所在的内存地址放进了列表。
当进入下一次循环时，代码执行了 trainer.zero_grad()。这步操作会原地（inplace）将那个内存地址里的数值清零。由于你列表里存的仅仅是那个内存地址的指针，所以列表里原先存好的数值也会跟着变成 0！
2. PyTorch 在后台维护着一个庞大的计算图（Computation Graph），用来记录所有的操作以便反向传播。如果只 clone，复制出来的新 Tensor 依然会附着在计算图上，PyTorch 会一直监控它，这会导致：